# U-Net sur le dataset des contours (version Kaggle)

Ce notebook reprend le meme enchainement que 097 : chargement du dataset, decoupage train / validation / test, entrainement, evaluation, sauvegarde du meilleur modele et visualisation des resultats.

Ici, on utilise un U-Net via segmentation-models-pytorch pour predire les masques binaires du dataset monte dans Kaggle.

In [ ]:
%pip install --quiet segmentation-models-pytorch matplotlib scikit-learn

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import segmentation_models_pytorch as smp  # pyright: ignore[reportMissingImports]

# ====================== CONFIGURATION ======================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

KAGGLE_ROOT = Path('/kaggle/input')
DATA_ROOT = None
if KAGGLE_ROOT.exists():
    for image_dir in KAGGLE_ROOT.rglob('train/imgs'):
        candidate = image_dir.parent.parent
        if (candidate / 'train' / 'labels').exists() and (candidate / 'test' / 'imgs').exists() and (candidate / 'test' / 'labels').exists():
            DATA_ROOT = candidate
            break

if DATA_ROOT is None:
    raise FileNotFoundError('Dataset not found in /kaggle/input. Expected train/imgs, train/labels, test/imgs and test/labels folders.')

TRAIN_IMG_DIR = DATA_ROOT / 'train' / 'imgs'
TRAIN_MASK_DIR = DATA_ROOT / 'train' / 'labels'
TEST_IMG_DIR = DATA_ROOT / 'test' / 'imgs'
TEST_MASK_DIR = DATA_ROOT / 'test' / 'labels'

OUTPUT_DIR = Path('/kaggle/working/outputs/unet_binary_segmentation')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 384
BATCH_SIZE = 8
EPOCHS = 45
VAL_FRACTION = 0.2
LEARNING_RATE =  2e-3
WEIGHT_DECAY = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('CUDA available:', torch.cuda.is_available())
print('Dataset root:', DATA_ROOT.resolve())
print('Output dir:', OUTPUT_DIR.resolve())

In [ ]:
def build_pairs(image_dir, mask_dir):
    image_paths = sorted(image_dir.glob('*.png'))
    pairs = [(image_path, mask_dir / image_path.name) for image_path in image_paths]
    missing_masks = [mask_path for _, mask_path in pairs if not mask_path.exists()]
    if missing_masks:
        raise FileNotFoundError(f'Missing mask file: {missing_masks[0]}')
    return pairs

train_pairs_full = build_pairs(TRAIN_IMG_DIR, TRAIN_MASK_DIR)
test_pairs = build_pairs(TEST_IMG_DIR, TEST_MASK_DIR)

train_pairs, val_pairs = train_test_split(
    train_pairs_full,
    test_size=VAL_FRACTION,
    random_state=SEED,
    shuffle=True,
)

print('Train pairs:', len(train_pairs))
print('Validation pairs:', len(val_pairs))
print('Test pairs:', len(test_pairs))
print('Sample image:', train_pairs[0][0].name)
print('Sample mask:', train_pairs[0][1].name)

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, pairs, image_size=256, augment=False):
        self.pairs = list(pairs)
        self.image_size = image_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]

        image = Image.open(image_path).convert('L').resize((self.image_size, self.image_size), Image.BILINEAR)
        mask = Image.open(mask_path).convert('L').resize((self.image_size, self.image_size), Image.NEAREST)

        image_array = np.asarray(image, dtype=np.float32) / 255.0
        mask_array = (np.asarray(mask, dtype=np.float32) < 128).astype(np.float32)

        if self.augment:
            if random.random() < 0.5:
                image_array = np.flip(image_array, axis=1).copy()
                mask_array = np.flip(mask_array, axis=1).copy()
            if random.random() < 0.25:
                image_array = np.flip(image_array, axis=0).copy()
                mask_array = np.flip(mask_array, axis=0).copy()

        image_tensor = torch.from_numpy(image_array).unsqueeze(0)
        mask_tensor = torch.from_numpy(mask_array).unsqueeze(0)
        return image_tensor, mask_tensor, image_path.name

train_dataset = SegmentationDataset(train_pairs, image_size=IMAGE_SIZE, augment=True)
val_dataset = SegmentationDataset(val_pairs, image_size=IMAGE_SIZE, augment=False)
test_dataset = SegmentationDataset(test_pairs, image_size=IMAGE_SIZE, augment=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

sample_image, sample_mask, sample_name = train_dataset[0]
print('Image tensor shape:', sample_image.shape)
print('Mask tensor shape:', sample_mask.shape)
print('Sample file:', sample_name)

In [ ]:
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights=None,
    in_channels=1,
    classes=1,
    activation=None,
).to(DEVICE)

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

def segmentation_loss(logits, masks):
    return 0.5 * bce_loss(logits, masks) + 0.5 * dice_loss(logits, masks)

def batch_metrics(logits, masks, threshold=0.5, eps=1e-7):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities > threshold).float()
    dims = (1, 2, 3)
    intersection = (predictions * masks).sum(dim=dims)
    prediction_sum = predictions.sum(dim=dims)
    mask_sum = masks.sum(dim=dims)
    dice = (2.0 * intersection + eps) / (prediction_sum + mask_sum + eps)
    union = prediction_sum + mask_sum - intersection
    iou = (intersection + eps) / (union + eps)
    return dice.mean().item(), iou.mean().item()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

def run_epoch(loader, training):
    model.train(training)
    running_loss = 0.0
    running_dice = 0.0
    running_iou = 0.0
    num_examples = 0

    for images, masks, _ in loader:
        images = images.to(DEVICE)
        masks = masks.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = segmentation_loss(logits, masks)
            if training:
                loss.backward()
                optimizer.step()

        batch_size = images.size(0)
        dice, iou = batch_metrics(logits.detach(), masks.detach())
        running_loss += loss.item() * batch_size
        running_dice += dice * batch_size
        running_iou += iou * batch_size
        num_examples += batch_size

    return {
        'loss': running_loss / num_examples,
        'dice': running_dice / num_examples,
        'iou': running_iou / num_examples,
    }

print('Model ready on:', DEVICE)

In [ ]:
history = {
    'train_loss': [],
    'train_dice': [],
    'train_iou': [],
    'val_loss': [],
    'val_dice': [],
    'val_iou': [],
}

best_val_dice = -1.0
best_model_path = OUTPUT_DIR / 'best_unet.pt'

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)
    scheduler.step(val_metrics['loss'])

    history['train_loss'].append(train_metrics['loss'])
    history['train_dice'].append(train_metrics['dice'])
    history['train_iou'].append(train_metrics['iou'])
    history['val_loss'].append(val_metrics['loss'])
    history['val_dice'].append(val_metrics['dice'])
    history['val_iou'].append(val_metrics['iou'])

    if val_metrics['dice'] > best_val_dice:
        best_val_dice = val_metrics['dice']
        torch.save(model.state_dict(), best_model_path)

    train_loss = train_metrics['loss']
    train_dice = train_metrics['dice']
    train_iou = train_metrics['iou']
    val_loss = val_metrics['loss']
    val_dice = val_metrics['dice']
    val_iou = val_metrics['iou']

    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'train_loss={train_loss:.4f} train_dice={train_dice:.4f} train_iou={train_iou:.4f} | '
        f'val_loss={val_loss:.4f} val_dice={val_dice:.4f} val_iou={val_iou:.4f}'
    )

print('Best validation Dice:', round(best_val_dice, 4))
print('Best model saved to:', best_model_path)

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
test_metrics = run_epoch(test_loader, training=False)

print('Test metrics:')
print(test_metrics)

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='train')
plt.plot(history['val_loss'], label='val')
plt.title('Loss')
plt.legend()

plt.subplot(1, 3, 2)
plt.plot(history['train_dice'], label='train')
plt.plot(history['val_dice'], label='val')
plt.title('Dice')
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(history['train_iou'], label='train')
plt.plot(history['val_iou'], label='val')
plt.title('IoU')
plt.legend()

plt.tight_layout()
plt.show()

@torch.no_grad()
def predict_mask(image_tensor):
    logits = model(image_tensor.unsqueeze(0).to(DEVICE))
    probabilities = torch.sigmoid(logits).cpu().squeeze(0).squeeze(0)
    return (probabilities > 0.5).float(), probabilities

sample_indices = np.random.default_rng(SEED).choice(len(test_dataset), size=min(6, len(test_dataset)), replace=False)
fig, axes = plt.subplots(len(sample_indices), 3, figsize=(10, 3 * len(sample_indices)))

if len(sample_indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_index, sample_index in enumerate(sample_indices):
    image_tensor, mask_tensor, sample_name = test_dataset[sample_index]
    predicted_mask, predicted_probabilities = predict_mask(image_tensor)

    image_array = image_tensor.squeeze(0).numpy()
    mask_array = mask_tensor.squeeze(0).numpy()
    predicted_array = predicted_mask.numpy()

    axes[row_index, 0].imshow(image_array, cmap='gray')
    axes[row_index, 0].set_title(f'Image: {sample_name}')
    axes[row_index, 0].axis('off')

    axes[row_index, 1].imshow(mask_array, cmap='gray')
    axes[row_index, 1].set_title('Masque reel')
    axes[row_index, 1].axis('off')

    axes[row_index, 2].imshow(predicted_array, cmap='gray')
    axes[row_index, 2].set_title('Prediction U-Net')
    axes[row_index, 2].axis('off')

plt.tight_layout()
plt.show()